In [6]:
from dinosaw import PEModel
from dinosaw.utils import load_image, normalize, resize_crop, do_2D_pca, convert_image, probe, closest_crop
from dinosaw.models.vit_wrapper import (
    PretrainedViTWrapper,
    MODEL_LIST,
)
import matplotlib.pyplot as plt
import cv2
import torch
from torchvision.transforms.functional import to_pil_image

In [7]:
vit_wrapper = PretrainedViTWrapper(MODEL_LIST[1], add_flash_attn=False, device="cuda").half()
#model = PEModel.load_from_checkpoint("/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/dinosaw/trained_models/test_long_train-epoch=468-val_loss=0.17.ckpt").half()
# Overfitted??
model = PEModel.load_from_checkpoint("/home/ab_aimd_anja_20884/Pawlowsky_Moritz/England/DINOMO/dinosaw/trained_models/test-epoch=67-val_loss=0.18.ckpt").half()

# Original image is 500x500 Pixels big how does the model perform when resizing to much larger size?

In [8]:
%%capture
img_size=1*518
mult_big=6

# slice_in = torch.tensor(cv2.imread("images/slice-In-Beam BSE-00001.jpg", cv2.IMREAD_GRAYSCALE)).repeat(3,1,1)
# print(slice_in.shape)
# test_img_small = convert_image(to_pil_image(slice_in), transform=resize_crop((img_size, img_size), (img_size, img_size)))
# test_img_big = convert_image(to_pil_image(slice_in), transform=resize_crop((img_size*mult_big, img_size*mult_big), (img_size*mult_big, img_size*mult_big)))
test_img_small = load_image("images/default_image.jpg", resize_crop((img_size, img_size), (img_size, img_size)))[0]
test_img_big = load_image("images/default_image.jpg", resize_crop((img_size*mult_big, img_size*mult_big), (img_size*mult_big, img_size*mult_big)))[0]



fig, ax = plt.subplots(1,2, figsize=(10,5))
ax[0].imshow(normalize(test_img_small.cpu().squeeze().permute(1,2,0).float()))
ax[1].imshow(normalize(test_img_big.cpu().squeeze().permute(1,2,0).float()))

In [9]:
res_small_dino = vit_wrapper.forward_features(test_img_small, make_2D=True)
res_big_dino = vit_wrapper.forward_features(test_img_big, make_2D=True)
res_small_our = model(test_img_small)
res_big_our = model(test_img_big)

In [10]:
%%capture
fig, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=300)
ax = ax.ravel()
ax[0].imshow(normalize(test_img_small.cpu().squeeze().permute(1,2,0).float()))
ax[0].set_title("Input")
ax[1].imshow(do_2D_pca(res_small_dino.squeeze(), post_norm="minmax"))
ax[1].set_title("DINOv2")
ax[2].imshow(do_2D_pca(res_small_our.squeeze(), post_norm="minmax"))
ax[2].set_title("Ours")
ax[3].imshow(normalize(test_img_big.cpu().squeeze().permute(1,2,0).float()))
ax[4].imshow(do_2D_pca(res_big_dino.squeeze(), post_norm="minmax"))
ax[5].imshow(do_2D_pca(res_big_our.squeeze(), n_components=3, post_norm="minmax"))

In [11]:
%%capture
for rmp in ["radial", "ud", "lr", "diag"]:
    probe([res_small_dino, res_small_our], [False]*2, ["DINO small", "Ours small"], ramp=rmp)
    probe([res_big_dino, res_big_our], [False]*2, ["DINO big", "Ours big"], ramp=rmp)

### No Positional Bias detected, but the semantics deteriorate

# How does it perform on naturally big images?

In [12]:
%%capture
img_size=1*518
mult_big=8
img_size_orig = (2300, 3000)

slice_in = torch.tensor(cv2.imread("images/AW_TIC_50Si_47SE_2kV39.tif", cv2.IMREAD_GRAYSCALE)).repeat(3,1,1)
print(slice_in.shape)
# test_img_small = convert_image(to_pil_image(slice_in), transform=resize_crop((img_size, img_size), (img_size, img_size)))
# test_img_big = convert_image(to_pil_image(slice_in), transform=resize_crop((img_size*mult_big, img_size*mult_big), (img_size*mult_big, img_size*mult_big)))
test_img_small = convert_image(to_pil_image(slice_in), transform=resize_crop((img_size, img_size), (img_size, img_size)))
test_img_big = convert_image(to_pil_image(slice_in), transform=closest_crop(2300, 2300))
test_img_big = convert_image(to_pil_image(slice_in), transform=resize_crop((2310, 2310), (2310, 2310)))
# test_img_small = load_image("images/default_image.jpg", resize_crop((img_size, img_size), (img_size, img_size)))[0]
# test_img_big = load_image("images/default_image.jpg", resize_crop((img_size*mult_big, img_size*mult_big), (img_size*mult_big, img_size*mult_big)))[0]



fig, ax = plt.subplots(1,2, figsize=(10,5))
ax[0].imshow(normalize(test_img_small.cpu().squeeze().permute(1,2,0).float()))
ax[1].imshow(normalize(test_img_big.cpu().squeeze().permute(1,2,0).float()))

In [13]:
res_small_dino = vit_wrapper.forward_features(test_img_small, make_2D=True)
res_big_dino = vit_wrapper.forward_features(test_img_big, make_2D=True)
res_small_our = model(test_img_small)
res_big_our = model(test_img_big)

In [14]:
%%capture
fig, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=300)
ax = ax.ravel()
ax[0].imshow(normalize(test_img_small.cpu().squeeze().permute(1,2,0).float()))
ax[0].set_title("Input")
ax[1].imshow(do_2D_pca(res_small_dino.squeeze(), post_norm="minmax"))
ax[1].set_title("DINOv2")
ax[2].imshow(do_2D_pca(res_small_our.squeeze(), post_norm="minmax"))
ax[2].set_title("Ours")
ax[3].imshow(normalize(test_img_big.cpu().squeeze().permute(1,2,0).float()))
ax[4].imshow(do_2D_pca(res_big_dino.squeeze(), post_norm="minmax"))
ax[5].imshow(do_2D_pca(res_big_our.squeeze(), post_norm="minmax"), cmap="grey")

In [15]:
%%capture
for rmp in ["radial", "ud", "lr", "diag"]:
    probe([res_small_dino, res_small_our], [False]*2, ["DINO small", "Ours small"], ramp=rmp)
    probe([res_big_dino, res_big_our], [False]*2, ["DINO big", "Ours big"], ramp=rmp)